<a href="https://colab.research.google.com/github/ShaikPasha22/LangChain_1_support_assist/blob/main/Copy_of_GitHub_Meeting_Action_Assignment_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Meeting Transcript → GitHub Issues (DIY Assignment)

This notebook builds the flow described in the assignment:

**`ChatPromptTemplate | tool-bound LLM | manual tool loop`**

Given a meeting transcript, the LLM extracts action items and calls a single
GitHub tool for every item it finds. The tool itself (not the LLM) decides
whether to actually create a GitHub issue, or just log the item for review.

Three production safeguards are implemented:
1. **Idempotency** — running the same `request_id` twice never creates a duplicate issue.
2. **Non-issue storage** — suggestions / tentative / unclear items are saved to SQLite, not dropped.
3. **Safe execution loop** — a `MAX_STEPS` cap + an allowlist of valid tool names.

> 📌 Read every comment before running — they explain *why*, not just *what*.


## 1. Install dependencies

Run this once per Colab session.

In [1]:
# LangChain core + the OpenAI chat model integration.
# (You can swap ChatOpenAI for any other LangChain chat model that supports
#  tool calling, e.g. ChatAnthropic or ChatGoogleGenerativeAI — the rest of
#  the notebook does not need to change.)
!pip install -q langchain langchain-core langchain-openai pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 16.8 MB/s eta 0:00:00


## 2. Imports

In [2]:
import sqlite3          # our simple "database" for idempotency + non-issue items
import json
from typing import Literal, Optional, List

from pydantic import BaseModel, Field          # for the tool's input schema
from langchain_core.tools import tool           # the @tool decorator
from langchain_core.messages import (
    SystemMessage, HumanMessage, AIMessage, ToolMessage
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


## 3. Configure your LLM API key

Enter your key when prompted (it is not stored anywhere in this notebook).

In [4]:
import os
from google.colab import userdata

# Reads your OpenRouter key from Colab Secrets (🔑 icon)
os.environ["OPEN_ROUTER_API_KEY"] = userdata.get("OPEN_ROUTER_API_KEY")

## 4. Set up the database

We use a single SQLite file with **one table** that stores every action item
we ever process — created issues *and* skipped/rejected items alike.

Why one table instead of two? Because idempotency and "save non-issue items"
are really the same lookup: "have I already handled `request_id` + this
action before, and what happened?"


In [5]:
DB_PATH = "meeting_actions.db"

def get_connection():
    """Return a fresh SQLite connection (Colab notebooks can be re-run out of order,
    so we always open/close a connection instead of keeping one global object around)."""
    return sqlite3.connect(DB_PATH)


def init_db():
    """Create the action_items table if it doesn't already exist.

    Columns follow the schema suggested in the assignment PDF, plus:
      - id: auto-increment primary key (just for our own bookkeeping)
      - reason: WHY an item was skipped/rejected (empty for created issues)
    """
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS action_items (
            id                  INTEGER PRIMARY KEY AUTOINCREMENT,
            request_id          TEXT NOT NULL,
            action_title        TEXT NOT NULL,
            owner               TEXT,
            commitment          TEXT,     -- confirmed / suggestion / tentative / unclear
            status              TEXT,     -- created / already_exists / skipped / rejected / failed
            reason              TEXT,
            github_issue_number INTEGER,
            github_issue_url    TEXT,
            UNIQUE(request_id, action_title)   -- <-- THIS is what makes idempotency easy
        )
    """)
    conn.commit()
    conn.close()

init_db()
print("Database ready ✅")


Database ready ✅


## 5. Pydantic input model for the GitHub tool

This defines **every field the LLM must fill in** when it calls the tool for
one action item. LangChain turns this schema into the JSON-schema the model
sees, so good field descriptions = better extractions.


In [6]:
class GitHubActionInput(BaseModel):
    """Schema for a single extracted action item from the meeting transcript."""

    request_id: str = Field(
        description="The stable request id for this whole meeting run. "
                    "Pass through the value given to you — never invent one."
    )
    title: str = Field(
        description="Short, clear title for the action item (used as the GitHub issue title)."
    )
    description: str = Field(
        description="1-3 sentence description of what needs to be done."
    )
    owner: Optional[str] = Field(
        default=None,
        description="Name of the person responsible, if mentioned in the transcript."
    )
    commitment_type: Literal["confirmed", "suggestion", "tentative", "unclear"] = Field(
        description=(
            "Classify how firm this action is:\n"
            "- 'confirmed': someone explicitly commits to doing it now "
            "(e.g. 'I will fix X').\n"
            "- 'suggestion': an idea floated with no firm commitment "
            "(e.g. 'we should consider X').\n"
            "- 'tentative': conditional / maybe-later commitment "
            "(e.g. 'I can do X if Y is ready').\n"
            "- 'unclear': you cannot confidently classify it."
        )
    )
    transcript_evidence: str = Field(
        description="The exact line(s) from the transcript that support this action item."
    )


## 6. The "provided" GitHub issue creator function

The assignment says this plain Python function is *given to you*. Below is a
stand-in implementation using the GitHub REST API directly (`requests`), so
the notebook is runnable end to end. **Replace the body of this function with
the exact one provided in your course materials if it differs** — everything
downstream only depends on its signature and return value.


In [7]:
import requests

# --- Fill these in with the repo + token provided in the assignment setup ---
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") #or getpass.getpass("Enter your GitHub Personal Access Token: ")
GITHUB_OWNER = "ShaikPasha22"   # <-- change me
GITHUB_REPO  = "LangChain_1_support_assist"                # <-- change me


def create_github_issue(title: str, body: str) -> dict:
    """Create a single GitHub issue.

    Returns a dict like {"number": 42, "html_url": "https://github.com/..."}
    Raises an exception if the API call fails — the caller (our tool) is
    responsible for catching that and recording a 'failed' status.
    """
    url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/issues"
    headers = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json",
    }
    payload = {"title": title, "body": body}
    response = requests.post(url, headers=headers, json=payload, timeout=15)
    response.raise_for_status()   # raises if GitHub returned an error status
    data = response.json()
    return {"number": data["number"], "html_url": data["html_url"]}


## 7. The GitHub LangChain tool

This is the ONLY tool the LLM is allowed to call. Note the key design choice:
**the LLM does not decide whether an issue gets created — this function does.**
The LLM's only job is to extract items and classify `commitment_type`
correctly; the tool enforces the business rule.

Logic inside the tool:
1. `unclear` commitment → **reject** (saved to DB, no issue).
2. `suggestion` / `tentative` → **skip** (saved to DB, no issue).
3. `confirmed` →
   - check the DB for `(request_id, title)` already existing → **already_exists** (idempotency!)
   - otherwise call `create_github_issue(...)` → **created**
   - if the GitHub API call throws → **failed** (saved to DB with the error as `reason`)


In [8]:
@tool("github_action_tool", args_schema=GitHubActionInput)
def github_action_tool(
    request_id: str,
    title: str,
    description: str,
    commitment_type: str,
    transcript_evidence: str,
    owner: Optional[str] = None,
) -> str:
    """Record a meeting action item. Creates a GitHub issue ONLY when the
    action is a confirmed commitment; otherwise saves it for later review.
    Call this once per action item you find in the transcript."""

    conn = get_connection()
    cur = conn.cursor()

    # --- Rule: unclear items are rejected outright, but still logged ---
    if commitment_type == "unclear":
        cur.execute(
            """INSERT OR IGNORE INTO action_items
               (request_id, action_title, owner, commitment, status, reason)
               VALUES (?, ?, ?, ?, ?, ?)""",
            (request_id, title, owner, commitment_type, "rejected",
             "Commitment type could not be classified confidently."),
        )
        conn.commit()
        conn.close()
        return f"REJECTED: '{title}' — commitment type unclear, saved for manual review."

    # --- Rule: suggestions / tentative items are saved, never turned into issues ---
    if commitment_type in ("suggestion", "tentative"):
        cur.execute(
            """INSERT OR IGNORE INTO action_items
               (request_id, action_title, owner, commitment, status, reason)
               VALUES (?, ?, ?, ?, ?, ?)""",
            (request_id, title, owner, commitment_type, "skipped",
             f"Not a confirmed commitment (classified as '{commitment_type}')."),
        )
        conn.commit()
        conn.close()
        return f"SKIPPED: '{title}' — {commitment_type}, saved to DB (no issue created)."

    # --- From here on, commitment_type == 'confirmed' ---

    # IDEMPOTENCY CHECK: has this exact (request_id, title) already been processed?
    cur.execute(
        "SELECT status, github_issue_number, github_issue_url FROM action_items "
        "WHERE request_id = ? AND action_title = ?",
        (request_id, title),
    )
    existing = cur.fetchone()
    if existing:
        status, issue_number, issue_url = existing
        conn.close()
        if status == "created":
            return (f"ALREADY_EXISTS: '{title}' was already created as issue "
                    f"#{issue_number} ({issue_url}). No duplicate created.")
        # If it previously failed/skipped for some reason, report that instead of retrying silently.
        return f"ALREADY_RECORDED: '{title}' already has status '{status}' for this request_id."

    # Not seen before -> actually create the GitHub issue.
    body = f"{description}\n\n---\nOwner: {owner or 'unassigned'}\nEvidence: {transcript_evidence}"
    try:
        issue = create_github_issue(title=title, body=body)
        cur.execute(
            """INSERT INTO action_items
               (request_id, action_title, owner, commitment, status, reason,
                github_issue_number, github_issue_url)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            (request_id, title, owner, commitment_type, "created", None,
             issue["number"], issue["html_url"]),
        )
        conn.commit()
        conn.close()
        return f"CREATED: issue #{issue['number']} for '{title}' -> {issue['html_url']}"
    except Exception as exc:
        # Save the failure too, so it isn't silently lost.
        cur.execute(
            """INSERT OR IGNORE INTO action_items
               (request_id, action_title, owner, commitment, status, reason)
               VALUES (?, ?, ?, ?, ?, ?)""",
            (request_id, title, owner, commitment_type, "failed", str(exc)),
        )
        conn.commit()
        conn.close()
        return f"FAILED: could not create issue for '{title}' — {exc}"


## 8. Prompt template + tool binding

`tools_repo1` is our allowlist of valid tools (just the one, here — but the
pattern supports more). `tool_llm1` is the LLM with that tool bound to it, so
it can emit `tool_calls` when it responds.


In [9]:
# The allowlist used by the safe execution loop (Rule 3 in the assignment).
tools_repo1 = [github_action_tool]
TOOLS_BY_NAME = {t.name: t for t in tools_repo1}

# --- LLM setup via OpenRouter (OpenAI-compatible endpoint) ---
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",              # OpenRouter's naming: "<provider>/<model>"
    api_key=os.environ["OPEN_ROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1", # points ChatOpenAI at OpenRouter instead of OpenAI
    temperature=0,
    default_headers={                        # optional, but OpenRouter recommends these
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "Meeting Transcript to GitHub Issues",
    },
)

# Base LLM + the same LLM with tools bound to it, so it can emit tool_calls.
tool_llm1 = llm.bind_tools(tools_repo1)

# The prompt instructs the model on HOW to read a transcript and what to call.
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You read meeting transcripts and extract every action item you find. "
     "For EACH action item (confirmed, suggestion, tentative, or unclear), "
     "call the `github_action_tool` exactly once with all required fields filled in. "
     "Always pass through the given request_id unchanged. "
     "Classify commitment_type carefully — do not mark something 'confirmed' unless "
     "a specific person explicitly commits to doing it. "
     "Do not create duplicate calls for the same action item. "
     "After you have called the tool for every action item you found, stop."),
    ("human",
     "request_id: {request_id}\n\nMeeting transcript:\n{transcript}"),
])

## 9. Safe manual tool-execution loop

This is the heart of the `ChatPromptTemplate | tool-bound LLM | manual tool loop`
pattern, plus **Rule 3 (safe execution)**:

- `MAX_STEPS` bounds how many back-and-forth rounds we allow (protects against
  infinite loops / runaway costs).
- Before invoking any tool, we check its name against `TOOLS_BY_NAME` — an
  unknown/hallucinated tool name is rejected instead of crashing the notebook.


In [10]:
MAX_STEPS = 6   # hard cap on how many LLM <-> tool round trips we allow

def run_meeting_to_issues(request_id: str, transcript: str):
    """Runs the full extract -> tool-call -> tool-result loop for one meeting.

    Returns the final list of chat messages (for debugging) — the actual
    results live in the SQLite DB, which we print separately afterwards.
    """
    # Build the initial message list from the prompt template.
    messages = prompt.format_messages(request_id=request_id, transcript=transcript)

    for step in range(1, MAX_STEPS + 1):
        ai_message: AIMessage = tool_llm1.invoke(messages)
        messages.append(ai_message)

        # If the model didn't ask for any tool calls, it's done.
        if not ai_message.tool_calls:
            print(f"[step {step}] Model finished — no more tool calls.")
            break

        print(f"[step {step}] Model requested {len(ai_message.tool_calls)} tool call(s).")

        # Execute every requested tool call and feed the results back as ToolMessages.
        for call in ai_message.tool_calls:
            tool_name = call["name"]
            tool_args = call["args"]
            call_id = call["id"]

            # --- SAFETY CHECK: reject any tool name not in our allowlist ---
            if tool_name not in TOOLS_BY_NAME:
                result_text = f"ERROR: '{tool_name}' is not an allowed tool. Call rejected."
                print("  ⚠️", result_text)
            else:
                result_text = TOOLS_BY_NAME[tool_name].invoke(tool_args)
                print("  ✅", result_text)

            messages.append(ToolMessage(content=str(result_text), tool_call_id=call_id))
    else:
        print(f"⚠️ Stopped after reaching MAX_STEPS={MAX_STEPS} — loop safety limit hit.")

    return messages


## 10. Sample input (from the assignment PDF)

In [11]:
request_id = "meeting-request-001"

meeting_transcript = """
Project: Customer Support Portal
Neha: Customers cannot reset passwords on mobile.
Arjun: I will fix the mobile password-reset issue.
Meera: We should consider adding WhatsApp login sometime.
Karan: I can test the reset flow if staging is ready.
Neha: Let us complete the password-reset fix first.
"""

# First run: should create 1 GitHub issue (Arjun's confirmed fix) and
# save the other two items (WhatsApp login = suggestion, testing = tentative).
final_messages = run_meeting_to_issues(request_id, meeting_transcript)


[step 1] Model requested 3 tool call(s).
  ✅ CREATED: issue #1 for 'Fix mobile password-reset issue' -> https://github.com/ShaikPasha22/LangChain_1_support_assist/issues/1
  ✅ SKIPPED: 'Consider adding WhatsApp login' — suggestion, saved to DB (no issue created).
  ✅ SKIPPED: 'Test password-reset flow' — tentative, saved to DB (no issue created).
[step 2] Model finished — no more tool calls.


## 11. Final results table

This satisfies "Return a clear result for every extracted item" — we just
read everything back out of SQLite for this `request_id`.


In [12]:
import pandas as pd

def show_results(request_id: str):
    conn = get_connection()
    df = pd.read_sql_query(
        "SELECT action_title, owner, commitment, status, reason, "
        "github_issue_number, github_issue_url FROM action_items "
        "WHERE request_id = ?",
        conn, params=(request_id,),
    )
    conn.close()
    return df

show_results(request_id)


,action_title,owner,commitment,status,reason,github_issue_number,github_issue_url
0,Consider adding WhatsApp login,None,suggestion,skipped,Not a confirmed commitment (classified as 'sug...,NaN,None
1,Fix mobile password-reset issue,Arjun,confirmed,created,None,1.0,https://github.com/ShaikPasha22/LangChain_1_su...
2,Test password-reset flow,Karan,tentative,skipped,Not a confirmed commitment (classified as 'ten...,NaN,None


## 12. Acceptance tests

Run each cell below and check the printed behaviour against the assignment's
acceptance criteria.


**Test 1 — re-run the SAME request_id: no duplicate issue should be created.**

In [13]:
# Expect: the confirmed item now returns ALREADY_EXISTS instead of creating a
# second GitHub issue. The suggestion/tentative rows are also just re-recorded,
# not duplicated, thanks to the UNIQUE(request_id, action_title) constraint.
_ = run_meeting_to_issues(request_id, meeting_transcript)
show_results(request_id)


[step 1] Model requested 3 tool call(s).
  ✅ ALREADY_EXISTS: 'Fix mobile password-reset issue' was already created as issue #1 (https://github.com/ShaikPasha22/LangChain_1_support_assist/issues/1). No duplicate created.
  ✅ SKIPPED: 'Consider adding WhatsApp login' — suggestion, saved to DB (no issue created).
  ✅ SKIPPED: 'Test password reset flow' — tentative, saved to DB (no issue created).
[step 2] Model finished — no more tool calls.


,action_title,owner,commitment,status,reason,github_issue_number,github_issue_url
0,Consider adding WhatsApp login,None,suggestion,skipped,Not a confirmed commitment (classified as 'sug...,NaN,None
1,Fix mobile password-reset issue,Arjun,confirmed,created,None,1.0,https://github.com/ShaikPasha22/LangChain_1_su...
2,Test password reset flow,Karan,tentative,skipped,Not a confirmed commitment (classified as 'ten...,NaN,None
3,Test password-reset flow,Karan,tentative,skipped,Not a confirmed commitment (classified as 'ten...,NaN,None


**Test 2 — a NEW request_id is treated as a brand-new run.**

In [14]:
new_request_id = "meeting-request-002"
_ = run_meeting_to_issues(new_request_id, meeting_transcript)
show_results(new_request_id)


[step 1] Model requested 3 tool call(s).
  ✅ CREATED: issue #2 for 'Fix mobile password-reset issue' -> https://github.com/ShaikPasha22/LangChain_1_support_assist/issues/2
  ✅ SKIPPED: 'Consider adding WhatsApp login' — suggestion, saved to DB (no issue created).
  ✅ SKIPPED: 'Test password-reset flow' — tentative, saved to DB (no issue created).
[step 2] Model finished — no more tool calls.


,action_title,owner,commitment,status,reason,github_issue_number,github_issue_url
0,Consider adding WhatsApp login,None,suggestion,skipped,Not a confirmed commitment (classified as 'sug...,NaN,None
1,Fix mobile password-reset issue,Arjun,confirmed,created,None,2.0,https://github.com/ShaikPasha22/LangChain_1_su...
2,Test password-reset flow,Karan,tentative,skipped,Not a confirmed commitment (classified as 'ten...,NaN,None


**Test 3 — unknown tool name is safely rejected (does not crash the loop).**

In [15]:
# We simulate the model calling a tool that isn't in our allowlist, by
# calling the loop's safety check directly rather than the whole LLM flow.
fake_call = {"name": "delete_repo_tool", "args": {}, "id": "fake123"}
if fake_call["name"] not in TOOLS_BY_NAME:
    print(f"ERROR: '{fake_call['name']}' is not an allowed tool. Call rejected.")
else:
    print("Unexpected: tool was allowed!")


ERROR: 'delete_repo_tool' is not an allowed tool. Call rejected.


**Test 4 — view every action item ever recorded, across all request_ids.**

In [16]:
conn = get_connection()
all_rows = pd.read_sql_query("SELECT * FROM action_items", conn)
conn.close()
all_rows


,id,request_id,action_title,owner,commitment,status,reason,github_issue_number,github_issue_url
0,1,meeting-request-001,Fix mobile password-reset issue,Arjun,confirmed,created,None,1.0,https://github.com/ShaikPasha22/LangChain_1_su...
1,2,meeting-request-001,Consider adding WhatsApp login,None,suggestion,skipped,Not a confirmed commitment (classified as 'sug...,NaN,None
2,3,meeting-request-001,Test password-reset flow,Karan,tentative,skipped,Not a confirmed commitment (classified as 'ten...,NaN,None
3,5,meeting-request-001,Test password reset flow,Karan,tentative,skipped,Not a confirmed commitment (classified as 'ten...,NaN,None
4,6,meeting-request-002,Fix mobile password-reset issue,Arjun,confirmed,created,None,2.0,https://github.com/ShaikPasha22/LangChain_1_su...
5,7,meeting-request-002,Consider adding WhatsApp login,None,suggestion,skipped,Not a confirmed commitment (classified as 'sug...,NaN,None
6,8,meeting-request-002,Test password-reset flow,Karan,tentative,skipped,Not a confirmed commitment (classified as 'ten...,NaN,None


## 13. Optional extension — email the owner

Sends a notification **only** when an issue was newly `created` (never for
`already_exists`, `skipped`, or `failed`). Email failure does not roll back
or duplicate the GitHub issue — it's reported separately, as required.

This uses a plain SMTP stub. Swap in the Gmail tool from your previous
session if you'd rather use that.


In [18]:
OWNER_EMAILS = {
    "Arjun": "zeeshpara@gmail.com",
      # added so our sample transcript's owner has a mapping
}

def send_owner_email(owner: str, issue_title: str, issue_number: int, issue_url: str) -> dict:
    """Stub email sender. Replace the body with your real Gmail tool call.

    Returns {'sent': True} on success or {'sent': False, 'error': str} on failure —
    this is intentionally returned as data, not raised, so a failed email can
    never accidentally trigger a retry of the GitHub issue creation above.
    """
    email = OWNER_EMAILS.get(owner)
    if not email:
        return {"sent": False, "error": f"No email mapping found for owner '{owner}'."}

    try:
        # --- Replace this block with your real Gmail-tool call ---
        print(f"(stub) Emailing {email}: issue #{issue_number} '{issue_title}' -> {issue_url}")
        # ----------------------------------------------------------
        return {"sent": True}
    except Exception as exc:
        return {"sent": False, "error": str(exc)}


def email_all_newly_created(request_id: str):
    """Loop over this request_id's rows and email owners of newly created issues only."""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute(
        "SELECT action_title, owner, github_issue_number, github_issue_url "
        "FROM action_items WHERE request_id = ? AND status = 'created'",
        (request_id,),
    )
    rows = cur.fetchall()
    conn.close()

    email_results = []
    for title, owner, issue_number, issue_url in rows:
        result = send_owner_email(owner, title, issue_number, issue_url)
        email_results.append({"title": title, "owner": owner, **result})
    return email_results

email_all_newly_created(request_id)


(stub) Emailing zeeshpara@gmail.com: issue #1 'Fix mobile password-reset issue' -> https://github.com/ShaikPasha22/LangChain_1_support_assist/issues/1


[{'title': 'Fix mobile password-reset issue', 'owner': 'Arjun', 'sent': True}]